In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader

import torchvision
import torchvision.transforms as transforms

import numpy as np
import matplotlib.pyplot as plt
from collections import defaultdict

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Используем: {device}")

Используем: cpu


In [ ]:
transform = transforms.Compose([transforms.ToTensor(), transforms.Normalize((0.2860,), (0.3530,))])
train_dataset = torchvision.datasets.FashionMNIST(root = './data', train = True, download = True, transform = transform)
test_dataset = torchvision.datasets.FashionMNIST(root = './data', train = False, download = True, transform = transform)
test_loader = DataLoader(test_dataset, batch_size = 64, shuffle = False)

print(f"Train size: {len(train_dataset)}")
print(f"Test size: {len(test_dataset)}")
print(f"Пример данных: {train_dataset[0][0].shape}, метка: {train_dataset[0][1]}")

In [ ]:
class PairDataset(Dataset):
    def __init__(self, base_dataset):
        self.base_dataset = base_dataset
        self.length = len(base_dataset)

        # Строим словарь: {класс: [список индексов]}
        self.class_to_indices = defaultdict(list)
        for idx, label in enumerate(base_dataset.targets.numpy()):
            self.class_to_indices[label].append(idx)

        # Список всех классов для удобства
        self.classes = list(self.class_to_indices.keys())

    def __len__(self):
        return self.length

    def __getitem__(self, _):
        # 1. Берем случайное изображение №1
        idx1 = np.random.randint(0, self.length)
        img1, label1 = self.base_dataset[idx1]

        if np.random.rand() < 0.5:
            # ПОЗИТИВНАЯ ПАРА
            # Ищем другой индекс с той же меткой
            candidates = self.class_to_indices[label1]
            idx2 = np.random.choice(candidates)

            # Защита от случая, когда idx2 == idx1 (та же картинка)
            while idx2 == idx1:
                idx2 = np.random.choice(candidates)

            img2, _ = self.base_dataset[idx2]
            pair_label = 0  # 0 = похожи

        else:
            # НЕГАТИВНАЯ ПАРА
            # Выбираем другой класс
            other_classes = [c for c in self.classes if c != label1]
            label2 = np.random.choice(other_classes)

            candidates = self.class_to_indices[label2]
            idx2 = np.random.choice(candidates)

            img2, _ = self.base_dataset[idx2]
            pair_label = 1  # 1 = разные

        return img1, img2, pair_label

pair_dataset = PairDataset(train_dataset)
pair_loader = DataLoader(pair_dataset, batch_size = 32, shuffle = True)

class Encoder(nn.Module):
    def __init__(self, embedding_dim = 128):
        super().__init__()

        self.conv = nn.Sequential(
            # Вход: (1, 28, 28)
            nn.Conv2d(1, 32, kernel_size = 3, padding = 1),   # (32, 28, 28)
            nn.ReLU(),
            nn.MaxPool2d(2),                              # (32, 14, 14)

            nn.Conv2d(32, 64, kernel_size = 3, padding = 1),  # (64, 14, 14)
            nn.ReLU(),
            nn.MaxPool2d(2),                              # (64, 7, 7)

            nn.Conv2d(64, 128, kernel_size = 3, padding = 1), # (128, 7, 7)
            nn.ReLU(),
            nn.MaxPool2d(kernel_size = 3, stride = 2, padding = 1)               # (128, 4, 4)
        )

        self.fc = nn.Sequential(
            nn.Flatten(),                  # → (128 * 4 * 4) = (2048,)
            nn.Linear(128 * 4 * 4, 256),   # → (256,)
            nn.ReLU(),
            nn.Linear(256, embedding_dim), # → (128,)
        )

    def forward(self, x):
        x = self.conv(x)
        x = self.fc(x)
        return x

class SiameseNetwork(nn.Module):
    def __init__(self, embedding_dim = 128):
        super().__init__()
        self.encoder = Encoder(embedding_dim)

    def forward(self, img1, img2):
        emb1 = self.encoder(img1)  # (batch, 128)
        emb2 = self.encoder(img2)  # (batch, 128)
        return emb1, emb2

class ContrastiveLoss(nn.Module):
    def __init__(self, margin = 1.0):
        super().__init__()
        self.margin = margin

    def forward(self, emb1, emb2, labels):
        # Считаем Евклидово расстояние между векторами
        dist = F.pairwise_distance(emb1, emb2)

        # Потери для похожих пар
        loss_pos = (1 - labels) * torch.pow(dist, 2)

        # Потери для разных пар
        loss_neg = labels * torch.pow(torch.clamp(self.margin - dist, min = 0.0), 2)

        loss = torch.mean(loss_pos + loss_neg)
        return loss

model = SiameseNetwork(embedding_dim = 128).to(device)
criterion = ContrastiveLoss(margin = 1.0)
optimizer = optim.Adam(model.parameters(), lr=1e-3)
num_epochs = 10
losses_history = []

for epoch in range(num_epochs):
    model.train()
    running_loss = 0.0
    num_batches = 0

    for imgs1, imgs2, labels in pair_loader:
        imgs1 = imgs1.to(device)
        imgs2 = imgs2.to(device)
        labels = labels.float().to(device)

        optimizer.zero_grad()

        emb1, emb2 = model(imgs1, imgs2)

        loss = criterion(emb1, emb2, labels)

        loss.backward()

        optimizer.step()

        running_loss += loss.item()
        num_batches += 1

    # Средний loss на эпохе
    avg_loss = running_loss / num_batches
    losses_history.append(avg_loss)

    print(f"Epoch [{epoch+1}/{num_epochs}] | Loss: {avg_loss:.4f}")


plt.figure(figsize=(10, 5))
plt.plot(range(1, num_epochs + 1), losses_history, marker='o', linewidth=2, color='steelblue')
plt.xlabel('Эпоха')
plt.ylabel('Loss')
plt.title('Динамика обучения Siamese Network')
plt.grid(True, alpha=0.3)
plt.xticks(range(1, num_epochs + 1))
plt.show()

In [ ]:
model.eval()

all_embeddings = []
all_labels = []

with torch.no_grad():
    for imgs, labels in test_loader:
        imgs = imgs.to(device)
        embs = model.encoder(imgs)
        all_embeddings.append(embs.cpu())
        all_labels.append(labels)

# Склеиваем все батчи в один большой тензор
all_embeddings = torch.cat(all_embeddings, dim=0)   # (10000, 128)
all_labels = torch.cat(all_labels, dim=0)           # (10000,)

print(f"Эмбеддинги: {all_embeddings.shape}")
print(f"Метки: {all_labels.shape}")
print(f"Первые 5 меток: {all_labels[:5]}")

def generate_pairs(embeddings, labels, num_pairs=5000):
    pairs = []
    pair_labels = []
    for _ in range(num_pairs):
        idx1 = np.random.randint(0, len(labels) - 1)
        label1 = labels[idx1].item()

        if np.random.rand() < 0.5:
            # Позитивная
            same_idx = (labels == label1).nonzero(as_tuple=True)[0]
            idx2 = same_idx[np.random.randint(0, len(same_idx) - 1)].item()
            while idx2 == idx1:
                idx2 = same_idx[np.random.randint(0, len(same_idx) - 1)].item()
            pair_labels.append(0)
        else:
            # Негативная
            other_idx = (labels != label1).nonzero(as_tuple=True)[0]
            idx2 = other_idx[np.random.randint(0, len(other_idx) - 1)].item()
            pair_labels.append(1)
        pairs.append((idx1, idx2))
    return pairs, torch.tensor(pair_labels)

test_pairs, test_pair_labels = generate_pairs(all_embeddings, all_labels, num_pairs=5000)

cos_sims = []
for idx1, idx2 in test_pairs:
    e1 = all_embeddings[idx1].unsqueeze(0)
    e2 = all_embeddings[idx2].unsqueeze(0)
    cos_sims.append(F.cosine_similarity(e1, e2).item())
cos_sims = torch.tensor(cos_sims)

THRESHOLD = 0.5
predictions = (cos_sims <= THRESHOLD).long()   # 0=похожи, 1=разные

accuracy = (predictions == test_pair_labels).float().mean().item()

print(f"Accuracy: {accuracy * 100:.2f}%")

In [ ]:
model.eval()

def get_emb(img_tensor):
    with torch.no_grad():
        img = img_tensor.unsqueeze(0).to(device)
        emb = model.encoder(img)
    return emb.squeeze(0).cpu()


num_examples = 5
fig, axes = plt.subplots(num_examples, 3, figsize=(13, 3 * num_examples))

for i in range(num_examples):
    idx1 = np.random.randint(0, len(test_dataset) - 1)
    label1 = test_dataset.targets[idx1].item()

    if np.random.rand() < 0.5:
        same_idx = (test_dataset.targets == label1).nonzero(as_tuple=True)[0]
        idx2 = same_idx[np.random.randint(0, len(same_idx) - 1)].item()
        while idx2 == idx1:
            idx2 = same_idx[np.random.randint(0, len(same_idx) - 1)].item()
        true_label = 0
    else:
        other_idx = (test_dataset.targets != label1).nonzero(as_tuple=True)[0]
        idx2 = other_idx[np.random.randint(0, len(other_idx) - 1)].item()
        true_label = 1

    img1, lab1 = test_dataset[idx1]
    img2, lab2 = test_dataset[idx2]

    emb1 = get_emb(img1)
    emb2 = get_emb(img2)

    cos = F.cosine_similarity(emb1.unsqueeze(0), emb2.unsqueeze(0)).item()

    pred = 0 if cos > THRESHOLD else 1

    img1_disp = img1.squeeze() * 0.3530 + 0.2860
    img2_disp = img2.squeeze() * 0.3530 + 0.2860

    axes[i, 0].imshow(img1_disp, cmap='gray')
    axes[i, 0].set_title(f"Img1: {test_dataset.classes[lab1]}")
    axes[i, 0].axis('off')

    axes[i, 1].imshow(img2_disp, cmap='gray')
    axes[i, 1].set_title(f"Img2: {test_dataset.classes[lab2]}")
    axes[i, 1].axis('off')

    axes[i, 2].axis('off')
    truth_str = "ПОХОЖИ" if true_label == 0 else "РАЗНЫЕ"
    pred_str = "ПОХОЖИ" if pred == 0 else "РАЗНЫЕ"

    color = 'green' if pred == true_label else 'red'
    status = "Правильно" if pred == true_label else "Ошибка"

    axes[i, 2].text(0.0, 0.8, f"Истина:  {truth_str}", fontsize=12, fontweight='bold')
    axes[i, 2].text(0.0, 0.55, f"Модель: {pred_str}", fontsize=12, color=color, fontweight='bold')
    axes[i, 2].text(0.0, 0.3, f"Косинус: {cos:.3f}", fontsize=11)
    axes[i, 2].text(0.0, 0.05, status, fontsize=13, color=color)

plt.tight_layout()
plt.show()